# 第23课 · 闭着眼也能找到最陡的上坡——梯度（gradient）与偏导（∂f/∂x）拼成的方向向量

**目标**：多变量函数对每个变量分别求导（偏导，partial derivative，∂f/∂x），拼成一个向量=梯度。梯度指向函数上升最快的方向。

**为什么对 Aurora 重要**：反向传播（`backward`）计算损失对每个权重的偏导，拼成梯度向量（gradient vector）；优化器用这个向量做一步参数更新（`optimizer.step()`）。


← **上一课**　[L22 · 导数](L22_derivatives.ipynb)

> 上节课学习了 **导数**：切线斜率、极限定义、数值微分 vs 解析微分。  
> 本课将探讨 **梯度**。

## 本课剧情：损失函数的最陡下坡在哪里？

想象你站在一座山上，闭着眼睛想找最快下山的路。  
你的脚能感受到"每个方向的坡度"——梯度（gradient）就是把这些坡度打包成一个向量。

对于多元函数（multivariate function） f(x₁, x₂, ..., xₙ)，梯度是：

$$\nabla f = \left(\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \ldots, \frac{\partial f}{\partial x_n}\right)$$

每个分量 ∂f/∂xᵢ 告诉你：**固定其他所有变量，只沿 xᵢ 方向移动一点，函数值怎么变？**

梯度向量指向**函数值增加最快**的方向（最陡上坡）。  
要让损失函数下降，就沿梯度的**反方向**走——这就是梯度下降（L25）的全部逻辑。

本课用中心差分（central difference）对每个分量数值化：

```python
∇f[i] ≈ (f(point + hᵢ) - f(point - hᵢ)) / (2h)
```

实现 `gradient(f, point, h=1e-5)`，对 Aurora 的任意损失函数都能用。

## 1. 偏导数：一次只问一个方向

**类比**：你是一个盲人厨师，想知道"汤的味道随盐量变化有多快"。  
你固定其他所有配料（水量、温度、火候），只改变盐量——这就是偏导数：

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x+h, y) - f(x-h, y)}{2h}$$

**手算例子**：f(x, y) = x² + y²

- ∂f/∂x：把 y 视为常数，对 x 求导 → ∂f/∂x = 2x
- ∂f/∂y：把 x 视为常数，对 y 求导 → ∂f/∂y = 2y

在点 (3, 4) 处：∇f(3,4) = [2×3, 2×4] = [6, 8]

**梯度的方向是什么？**  
梯度向量 [6, 8] 垂直于等高线（contour line）f(x,y)=25 在 (3,4) 处的切线，指向函数值增大最快的方向。

> 实现 `gradient(f, point)` 时：对每个分量 i，构造 ±h 的微扰向量，做中心差分。

## 实验入口：用数值变化观察函数

观察 `dfdx` 和 `dfdy` 在点 `(3, 4)` 处的值，确认中心差分结果与解析公式 `∂f/∂x = 2x = 6`、`∂f/∂y = 2y = 8` 的误差在 1e-9 以内。


In [ ]:
import numpy as np
f = lambda p: p[0]**2 + p[1]**2     # f(x,y) = x^2 + y^2
h = 1e-5
p = np.array([3.0, 4.0])
dfdx = (f(p+[h,0]) - f(p-[h,0]))/(2*h)
dfdy = (f(p+[0,h]) - f(p-[0,h]))/(2*h)
print('∂f/∂x ≈', round(dfdx,3), '(真值2x=6)  ∂f/∂y ≈', round(dfdy,3), '(真值2y=8)')

## 动手观察：变化率不是一句口号

对一元函数 `f(x) = x²` 在五个点上分别用中心差分估计斜率，打印结果与解析值 `f'(x) = 2x` 对比，直观感受 h=1e-3 时的数值精度。


In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

遍历 `x ∈ [-3, 3]` 的七个点，打印每处的函数值与数值斜率，确认斜率始终接近解析导数 `f'(x) = 2x + 2`。


In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现 `gradient(f, point, h=1e-5)`

对每个分量做一次中心差分，返回梯度向量。

**推理路线**：
1. 初始化 `grad = np.zeros_like(point)`，确保输出 shape 与输入相同
2. 对每个维度 i，构造单位扰动向量 `e`：`e = np.zeros_like(point); e[i] = 1.0`，其余分量为 0
3. 计算 `(f(point + e*h) - f(point - e*h)) / (2 * h)`，存入 `grad[i]`（两种写法等价，本课统一用单位向量乘 h 的形式，与下方变量表一致）
4. 关键点：每次只扰动第 i 维，其余维度不变——这正是偏导数"固定其他变量"的操作

**参考输入输出**：f(x, y) = x² + y², point = [3, 4] → ∂f/∂x = 2×3 = 6, ∂f/∂y = 2×4 = 8 → 梯度 = [6, 8]

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `gradient` 前明确三件事：
- 输入：`f`（可调用函数）、`point`（n 维 numpy 数组）、`h`（步长，默认 1e-5）
- 关键步骤：对每个维度 i 构造单位扰动向量 `e`，计算 `(f(p + e*h) - f(p - e*h)) / (2h)`
- 返回：长度为 n 的 numpy 数组，第 i 个元素是 ∂f/∂xᵢ 在 `point` 处的值


In [ ]:
def gradient(f, point, h=1e-5):
    point = np.asarray(point, dtype=float)
    grad = np.zeros_like(point)
    # ✏️ TODO: 填满 grad 的每个分量
    raise NotImplementedError("TODO: 逐分量计算偏导数 ∂f/∂xᵢ = (f(x+ε·eᵢ) - f(x-ε·eᵢ)) / (2ε)")

In [ ]:
g = gradient(lambda p: p[0]**2 + p[1]**2, [3.0, 4.0])
print('梯度 =', np.round(g, 3), '(应≈ [6, 8])')
assert np.allclose(g, [6.0, 8.0], atol=1e-4)
print('✅ 通过：你会算梯度了。')

**🔗 Aurora 连接**：训练时的损失函数（loss function） `L(W₁, W₂, …)` 是所有权重的多元函数；`backward()` 计算的正是该函数的梯度，每个权重对应梯度向量的一个分量。Aurora 尚未实现解析反向传播；本课的数值梯度是其出发点；反向传播用链式法则（chain rule）代替一个参数一个参数地数值试探，把 2n 次前向传播压缩成 1 次前向 + 1 次后向；参数越多，两者的速度差距就越大。下一课讲链式法则，解释这个压缩的来源。

## 参数实验：梯度方向即等高线（contour line）法向量

下方代码在 point = [3, 4] 处验证 `gradient`。照着它把 point 换成 [1, 2]，调用 `gradient(f, [1.0, 2.0])` 应得到 [2, 4]（解析值：∂f/∂x = 2×1 = 2, ∂f/∂y = 2×2 = 4）。

验证梯度方向的几何含义：f(x, y) = x² + y² 的等高线是以原点为圆心的同心圆；点 (1, 2) 处的等高线切线方向是 (-2, 1)（与梯度 (2, 4) 点积为零），而梯度方向 (2, 4) 指向圆心外侧，即函数值增大最快的方向。梯度取负号得 (-2, -4)，指向圆心——这正是梯度下降每步的移动方向。

In [ ]:
# 参数实验：梯度方向 = 等高线法向量
f_2d = lambda p: p[0]**2 + p[1]**2

point = [3.0, 4.0]
g = gradient(f_2d, point)
print(f'gradient(x²+y², [3,4]) = {np.round(g, 6)}')
print(f'解析值 [2×3, 2×4]      = [6, 8]')
assert np.allclose(g, [6.0, 8.0], atol=1e-4), f'梯度不对: {g}'
print('✅ 数值梯度与解析值吻合')

# 梯度方向验证：f(x,y)=x²+y² 的梯度在点 p 处应与 p 方向一致（径向朝外）
# 说明：tangent=[-g[1],g[0]] 的点积恒为 -ab+ba=0（代数恒等式），与实现无关，不能检验正确性。
# 改用解析径向方向比较，才能真正检验 gradient() 是否算对。
grad_dir = g / np.linalg.norm(g)
radial = np.array(point, dtype=float) / np.linalg.norm(np.array(point, dtype=float))
assert np.allclose(grad_dir, radial, atol=1e-4), \
    f"梯度方向不对: {grad_dir}，期望径向方向: {radial}"
print(f'✅ 梯度方向与解析径向方向吻合 (grad_dir≈{np.round(grad_dir,4)}, radial={np.round(radial,4)})')


In [ ]:
# 三元函数梯度：f(x,y,z) = x² + 2y² + 3z² 在 (1,1,1)
f_3d = lambda p: p[0]**2 + 2*p[1]**2 + 3*p[2]**2
g3 = gradient(f_3d, [1.0, 1.0, 1.0])
print(f'gradient(x²+2y²+3z², [1,1,1]) = {np.round(g3, 6)}')
print(f'解析值                          = [2, 4, 6]')
assert np.allclose(g3, [2.0, 4.0, 6.0], atol=1e-4)
print('✅ 三元函数梯度验证通过')
print(f'梯度范数 = {np.linalg.norm(g3):.4f}（下降最陡方向的步长缩放因子）')


## 本课收束

现在可以调用 `gradient(f, point)` 对任意多元函数在任意点求数值梯度（numerical gradient），返回与 `point` 等长的偏导向量。梯度的每一维对应一个输入变量，指向使 `f` 增大最快的方向；训练时取负号即得下降方向。Aurora 的 `backward` 计算的是同一个向量，只是用解析推导代替数值扰动，速度快几个数量级。

下一课：**L24**（链式法则）讲复合函数的梯度为何等于各层偏导的乘积。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：梯度手算（目标 8 分钟）

盖上屏幕，纸上作答：

**问 1**：f(x, y) = x² + y²，写出 ∂f/∂x 和 ∂f/∂y 的解析公式。

**问 2**：用问 1 的公式，计算 ∇f(3, 4) = ?

**问 3**：f(x, y, z) = x² + 2y² + 3z²，在点 (1, 1, 1) 处的梯度 ∇f = ?

**问 4**：梯度向量的方向有什么几何含义？  
（提示：它与等高线（contour line）的关系是什么？）

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：∂f/∂x = 2x, ∂f/∂y = 2y
f_2d = lambda p: p[0]**2 + p[1]**2
dfdx = lambda p: 2 * p[0]
dfdy = lambda p: 2 * p[1]
p = np.array([3., 4.])
assert np.isclose(dfdx(p), 6.0) and np.isclose(dfdy(p), 8.0)
print(f"Q1 ✅  ∂f/∂x=2x, ∂f/∂y=2y  （在({p[0]:.0f},{p[1]:.0f})处分别为{dfdx(p):.0f},{dfdy(p):.0f}）")

# 问2：∇f(3,4) = [6, 8]
grad_analytical = np.array([dfdx(p), dfdy(p)])
assert np.allclose(grad_analytical, [6., 8.], atol=1e-10)
try:
    grad_num = gradient(f_2d, p)
    assert np.allclose(grad_num, grad_analytical, atol=1e-7), \
        f"数值梯度 {grad_num} 与解析值 {grad_analytical} 不符"
    print(f"Q2 ✅  ∇f(3,4)={grad_analytical}，数值验证={np.round(grad_num,6)}")
except NotImplementedError:
    print("⬜ Q2：请先实现 gradient()，再运行对答案格")

# 问3：∇f(1,1,1) = [2, 4, 6] for f=x²+2y²+3z²
f_3d = lambda p: p[0]**2 + 2*p[1]**2 + 3*p[2]**2
q3 = np.array([1., 1., 1.])
grad_3d_analytical = np.array([2., 4., 6.])
try:
    grad_3d = gradient(f_3d, q3)
    assert np.allclose(grad_3d, grad_3d_analytical, atol=1e-7)
    print(f"Q3 ✅  ∇f(1,1,1)={grad_3d_analytical}，数值={np.round(grad_3d,6)}")
except NotImplementedError:
    print("⬜ Q3：请先实现 gradient()，再运行对答案格")

# 问4：梯度几何含义（数值验证：梯度⊥等高线切线）
# 在 (3,4) 处，等高线 x²+y²=25 的切线方向是 (-y, x)/||...||= (-4,3)/5
tangent = np.array([-4., 3.]) / 5.0
grad_norm = grad_analytical / np.linalg.norm(grad_analytical)
dot = abs(np.dot(tangent, grad_norm))
assert dot < 1e-10, f"梯度应垂直于等高线切线，内积={dot}"
print(f"Q4 ✅  ∇f(3,4)=[6,8] 垂直于等高线切线(-4,3)，内积={dot:.2e}")
print("     梯度 = 函数值增加最快的方向 = 等高线法向量")
print("\n🎉 梯度白板挑战通过！多元函数最陡上坡的概念已内化。")

In [ ]:
# ✏️ 本课自评
l23_review = {
    "partial_derivative_formula":  None,  # 理解 ∂f/∂xᵢ 的偏导定义？True/False
    "gradient_implemented":        None,  # gradient(f, point) 实现并通过断言？True/False
    "gradient_direction_meaning":  None,  # 梯度=最陡上坡=等高线法向？True/False
    "3d_gradient_computed":        None,  # 能计算 3 元函数的梯度？True/False
    "whiteboard_passed":           None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l23_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l23_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L23 全部通关！进入 L24：链式法则')

---

→ **下一课**　[L24 · 链式法则](L24_chain_rule.ipynb)

> 下节课将学习 **链式法则**：函数套函数的求导，反向传播的数学基础。